# highlevel

> High-level async completion/stream wrappers with endpoint dispatch.

In [ ]:
#| default_exp highlevel


## Notebook Overview

This notebook documents `fastllm_v2.highlevel` in nbdev style, interleaving explanations, source cells, and lightweight REPL checks.

- **Depends on:** `clients, config, types`
- **Primary symbols:** `_ENDPOINTS, _host, _model_name, infer_provider, _resolve_api_key, mk_auto_client, _coerce_part, _coerce_msg, _coerce_messages, _missing_responses_endpoint, _resolve_endpoint, _merge_aliases`


## Module Setup

Imports, constants, and shared setup used by the symbols below.


In [ ]:
#| export
"High-level completion wrappers with automatic client routing."

from __future__ import annotations

import os
from typing import Any, Optional
from urllib.parse import urlparse

import httpx
from fastcore.meta import delegates

from .clients import AnthropicClient, GeminiClient, OpenAIClient
from .config import ClientConfig
from .types import Msg, Part, RequestOptions


_ENDPOINTS = ("auto", "responses", "chat")


### `_host`

Implementation for `_host` in the `highlevel` module.


In [ ]:
#| export


def _host(base_url: str = "") -> str: return (urlparse(base_url or "").hostname or "").lower()


### `_model_name`

Implementation for `_model_name` in the `highlevel` module.


In [ ]:
#| export


def _model_name(model: str = "") -> str:
    m = (model or "").strip().lower()
    for pref in ("anthropic/", "google/", "openai/"):
        if m.startswith(pref): return m.split("/", 1)[1]
    return m


### `infer_provider`

Infer provider family from base URL, then model prefix.


In [ ]:
#| export


def infer_provider(model: str, *, provider: str = "", base_url: str = "") -> str:
    "Infer provider family from base URL, then model prefix."
    _ = provider  # Backward-compatible arg; routing no longer depends on explicit provider.

    host = _host(base_url)
    if "anthropic.com" in host: return "anthropic"
    if "generativelanguage.googleapis.com" in host: return "gemini"
    if host == "api.openai.com": return "openai"

    m = _model_name(model)
    if m.startswith("claude") or "claude-" in m: return "anthropic"
    if m.startswith("gemini") or m.startswith("models/gemini") or "gemini-" in m: return "gemini"
    if m.startswith("gpt"): return "openai"
    return "openai_compat"


### `_resolve_api_key`

Implementation for `_resolve_api_key` in the `highlevel` module.


In [ ]:
#| export


def _resolve_api_key(family: str, *, api_key: str = "") -> str:
    if api_key: return api_key
    if family == "anthropic": return os.getenv("ANTHROPIC_API_KEY") or ""
    if family == "gemini": return os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or ""
    return os.getenv("OPENAI_API_KEY") or ""


### `mk_auto_client`

Create a provider client from model/base_url inference.


In [ ]:
#| export


def mk_auto_client(model: str, *, api_key: str = "", base_url: str = "", provider: str = "", timeout: float = 60.0):
    "Create a provider client from model/base_url inference."
    _ = provider  # Backward-compatible arg; provider hooks removed.
    family = infer_provider(model, base_url=base_url)
    key = _resolve_api_key(family, api_key=api_key)
    resolved_provider = "openai" if family == "openai" else ("openai_compat" if family == "openai_compat" else family)
    cfg = ClientConfig(model=model, api_key=key, base_url=base_url, provider=resolved_provider, timeout=timeout)
    if family == "anthropic": return AnthropicClient(cfg)
    if family == "gemini": return GeminiClient(cfg)
    return OpenAIClient(cfg)


### `_coerce_part`

Implementation for `_coerce_part` in the `highlevel` module.


In [ ]:
#| export


def _coerce_part(p: Any) -> Part:
    if isinstance(p, Part): return p
    if p is None: return Part(type="text", text="")
    if isinstance(p, str): return Part(type="text", text=p)
    if isinstance(p, dict):
        d = dict(p)
        typ = str(d.pop("type", "text"))
        txt = d.pop("text", None)
        if txt is None and typ == "text" and isinstance(d.get("content"), str): txt = d.pop("content")
        return Part(type=typ, text=txt, data=(d or None))
    return Part(type="text", text=str(p))


### `_coerce_msg`

Implementation for `_coerce_msg` in the `highlevel` module.


In [ ]:
#| export


def _coerce_msg(m: Any) -> Msg:
    if isinstance(m, Msg): return m
    if isinstance(m, str): return Msg(role="user", content=[Part(type="text", text=m)])
    if isinstance(m, dict):
        d = dict(m)
        role = str(d.pop("role", "user"))
        content = d.pop("content", "")
        if isinstance(content, list): parts = [_coerce_part(o) for o in content]
        elif content in (None, ""): parts = []
        else: parts = [_coerce_part(content)]
        if not parts and role != "assistant": parts = [Part(type="text", text="")]
        return Msg(role=role, content=parts, data=(d or None))
    raise TypeError(f"Unsupported message type: {type(m).__name__}")


### `_coerce_messages`

Implementation for `_coerce_messages` in the `highlevel` module.


In [ ]:
#| export


def _coerce_messages(messages: Any) -> list[Msg]:
    if isinstance(messages, (str, Msg, dict)): return [_coerce_msg(messages)]
    if isinstance(messages, list) or isinstance(messages, tuple): return [_coerce_msg(m) for m in messages]
    return [_coerce_msg(messages)]


### `_missing_responses_endpoint`

Implementation for `_missing_responses_endpoint` in the `highlevel` module.


In [ ]:
#| export


def _missing_responses_endpoint(e: Exception) -> bool:
    if isinstance(e, AttributeError): return True
    if not isinstance(e, httpx.HTTPStatusError): return False
    code, txt = e.response.status_code, (e.response.text or "").lower()
    if code in (404, 405, 410, 501): return True
    if code in (400, 422) and "responses" in txt and any(k in txt for k in ("not found", "unknown", "unsupported", "invalid")):
        return True
    return False


### `_resolve_endpoint`

Implementation for `_resolve_endpoint` in the `highlevel` module.


In [ ]:
#| export


def _resolve_endpoint(family: str, endpoint: str) -> str:
    if endpoint not in _ENDPOINTS: raise ValueError(f"Invalid endpoint='{endpoint}', expected one of {_ENDPOINTS}")
    if family in ("anthropic", "gemini"): return endpoint
    if endpoint == "auto": return "responses" if family == "openai" else "chat"
    return endpoint


### `_merge_aliases`

Implementation for `_merge_aliases` in the `highlevel` module.


In [ ]:
#| export


def _merge_aliases(kwargs: dict) -> dict:
    kw = dict(kwargs)
    if "caching" in kw and "cache" not in kw: kw["cache"] = kw.pop("caching")
    if "api_base" in kw and "base_url" not in kw: kw["base_url"] = kw.pop("api_base")
    kw.pop("provider", None)
    kw.pop("custom_llm_provider", None)
    return kw


### `_openai_complete`

Implementation for `_openai_complete` in the `highlevel` module.


In [ ]:
#| export


async def _openai_complete(c: OpenAIClient, msgs: list[Msg], opts: Optional[RequestOptions], ep: str, fallback: bool, kw: dict):
    if ep == "chat": return await c.achat_complete(msgs, options=opts, **kw)
    try: return await c.acomplete(msgs, options=opts, **kw)
    except Exception as e:
        if not fallback or not _missing_responses_endpoint(e): raise
        return await c.achat_complete(msgs, options=opts, **kw)


### `_openai_stream`

Implementation for `_openai_stream` in the `highlevel` module.


In [ ]:
#| export


async def _openai_stream(c: OpenAIClient, msgs: list[Msg], opts: Optional[RequestOptions], ep: str, fallback: bool, kw: dict):
    if ep == "chat":
        async for d in c.achat_stream(msgs, options=opts, **kw): yield d
        return
    try:
        async for d in c.astream(msgs, options=opts, **kw): yield d
        return
    except Exception as e:
        if not fallback or not _missing_responses_endpoint(e): raise
    async for d in c.achat_stream(msgs, options=opts, **kw): yield d


### `acompletion`

LiteLLM-style async completion wrapper; returns an async iterator when `stream=True`.


In [ ]:
#| export


@delegates(RequestOptions, keep=True)
async def acompletion(model: str, messages: Any, *, stream: bool = False, api_key: str = "", base_url: str = "",
    api_base: str = "", provider: str = "", custom_llm_provider: str = "", endpoint: str = "auto",
    timeout: float = 60.0, options: Optional[RequestOptions] = None, **kwargs):
    "LiteLLM-style async completion wrapper; returns an async iterator when `stream=True`."
    if api_base and not base_url: base_url = api_base
    _ = provider, custom_llm_provider  # Backward-compatible args; routing is inferred from model/base_url.
    kw = _merge_aliases(kwargs)
    msgs = _coerce_messages(messages)
    family = infer_provider(model, base_url=base_url)
    ep = _resolve_endpoint(family, endpoint)
    c = mk_auto_client(model, api_key=api_key, base_url=base_url, timeout=timeout)
    fallback = family == "openai" and endpoint == "auto"

    if not stream:
        try:
            if family == "anthropic": return await c.acomplete(msgs, options=options, **kw)
            if family == "gemini": return await c.acomplete(msgs, options=options, **kw)
            return await _openai_complete(c, msgs, options, ep, fallback, kw)
        finally: await c.aclose()

    async def _gen():
        try:
            if family == "anthropic":
                async for d in c.astream(msgs, options=options, **kw): yield d
                return
            if family == "gemini":
                async for d in c.astream(msgs, options=options, **kw): yield d
                return
            async for d in _openai_stream(c, msgs, options, ep, fallback, kw): yield d
        finally: await c.aclose()

    return _gen()


### `astream`

Convenience async stream wrapper.


In [ ]:
#| export


@delegates(acompletion, keep=True, but=["stream"])
async def astream(model: str, messages: Any, **kwargs):
    "Convenience async stream wrapper."
    it = await acompletion(model, messages, stream=True, **kwargs)
    async for d in it: yield d


## Supporting Definitions

Additional constants or helpers that support later exports.


In [ ]:
#| export


__all__ = "infer_provider mk_auto_client acompletion astream".split()


## REPL Checks

Quick smoke checks to confirm exported symbols import correctly.


In [ ]:
#| hide
from fastcore.test import test
import fastllm_v2.highlevel as m

test(m is not None)
test(hasattr(m, '_ENDPOINTS'))
